In [5]:
"""
Demand forecasting model (Units Sold) using LightGBM.

Fixes applied vs. the original notebook version:
  1. Single-pass preprocessing only (no accidental double label-encoding /
     double StandardScaler fit — the original re-ran encoding/scaling on
     data that might already have been encoded/scaled by an earlier cell).
  2. Time-based train/test split instead of a random split, so the model
     is evaluated the way it will actually be used: predicting forward in
     time, never on rows whose date falls before rows in the training set.
  3. Categorical columns are explicitly declared to LightGBM
     (categorical_feature=...), so it treats them as categories rather
     than as ordered integers.
  4. Early stopping on a validation set, instead of a hardcoded
     n_estimators=1000 with no check on over/underfitting.
  5. The target (Units Sold) is trained and evaluated in its native
     units — no scale-then-inverse-transform round trip, which removes
     a whole class of "which scale is this number in" bugs.
  6. A StandardScaler is still fit (on the training split only, to avoid
     leakage) and saved separately, for use by *other* parts of your
     pipeline (e.g. the RL environment) that expect standardized inputs.
     It is not used by this model itself, since tree models don't need
     scaled features.

New in this version:
  7. Lag / rolling features per store-product series (yesterday's sales,
     7/14-day rolling mean & std of sales, previous day's inventory).
     These are computed with a shift(1) before any rolling window, so a
     row's features only ever use information strictly before that row's
     own date -- no leakage, and it's still safe to compute these before
     the train/test split since each row only looks backward within its
     own store-product group.
  8. A quantile-regression variant (LightGBM objective="quantile") in
     addition to the standard mean-regression model. Trained once per
     alpha in QUANTILE_ALPHAS. Use the high quantile (e.g. 0.9) as a
     safety-stock-oriented demand estimate: ordering enough to cover the
     90th-percentile forecast instead of just the average protects
     against underordering, which matters for a "can't accept stockout"
     policy.

Usage:
    python train_lgbm_demand_model.py

Edit CSV_PATH below to point at your file.
"""

import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------------
# 0. Config
# ------------------------------------------------------------------
CSV_PATH = "C:\\Users\\a0952\\Desktop\\數創\\archive (1)\\retail_store_inventory.csv"   # <-- edit this to wherever your CSV lives on your machine
TEST_FRACTION = 0.2                       # last 20% of dates -> test set
RANDOM_STATE = 42

TARGET_COL = "Units Sold"
GROUP_COLS = ["Store ID", "Product ID"]   # each store-product pair is its own time series
CATEGORICAL_COLS = ["Store ID", "Product ID", "Category", "Region",
                     "Weather Condition", "Seasonality"]
# Columns you may want standardized for downstream use (e.g. the RL env).
# Not used by the LightGBM model itself.
NUMERIC_COLS_FOR_SCALER = ["Inventory Level", "Units Sold", "Price", "Competitor Pricing"]

# Quantiles to train a separate model for, in addition to the mean model.
# 0.9 -> "cover the 90th percentile of demand", useful for a no-stockout policy.
QUANTILE_ALPHAS = [0.9]

OUTPUT_MEAN_MODEL_PATH = "lgb_sales_model_mean.pkl"
OUTPUT_QUANTILE_MODEL_PATH_TMPL = "lgb_sales_model_q{alpha}.pkl"   # e.g. lgb_sales_model_q0.9.pkl
OUTPUT_ENCODERS_PATH = "label_encoders.pkl"
OUTPUT_SCALER_PATH = "scaler.pkl"
OUTPUT_FEATURE_ORDER_PATH = "feature_order.pkl"


def load_and_engineer(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(GROUP_COLS + ["Date"]).reset_index(drop=True)

    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["DayOfWeek"] = df["Date"].dt.dayofweek
    return df


def add_lag_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add per-series lag/rolling features. Computed on the full sorted
    dataframe (before the train/test split) -- this is safe because
    groupby(...).shift/rolling only ever looks backward in time within
    each store-product's own history, regardless of where the train/test
    cutoff later falls.

    Rows at the start of each series won't have enough history for the
    longer windows and will get NaNs there -- those rows are dropped
    at the end (a small, one-time cost per series, not an ongoing one).
    """
    g = df.groupby(GROUP_COLS, observed=True)["Units Sold"]

    df["Sales_Lag_1"] = g.shift(1)
    df["Sales_Lag_7"] = g.shift(7)
    df["Sales_RollMean_7"] = g.transform(lambda s: s.shift(1).rolling(7).mean())
    df["Sales_RollStd_7"] = g.transform(lambda s: s.shift(1).rolling(7).std())
    df["Sales_RollMean_14"] = g.transform(lambda s: s.shift(1).rolling(14).mean())

    inv_g = df.groupby(GROUP_COLS, observed=True)["Inventory Level"]
    df["Inventory_Lag_1"] = inv_g.shift(1)

    before = len(df)
    df = df.dropna(subset=[
        "Sales_Lag_1", "Sales_Lag_7", "Sales_RollMean_7",
        "Sales_RollStd_7", "Sales_RollMean_14", "Inventory_Lag_1",
    ]).reset_index(drop=True)
    print(f"Dropped {before - len(df)} rows with insufficient history for lag/rolling features "
          f"(expected: ~14 rows per store-product series, only at the start of each series).")
    return df


def time_based_split(df: pd.DataFrame, test_fraction: float):
    """Split by date cutoff so test rows are strictly later in time than
    train rows -- avoids the leakage a random train_test_split introduces
    on time series data."""
    unique_dates = np.sort(df["Date"].unique())
    cutoff_idx = int(len(unique_dates) * (1 - test_fraction))
    cutoff_date = unique_dates[cutoff_idx]

    train_df = df[df["Date"] < cutoff_date].copy()
    test_df = df[df["Date"] >= cutoff_date].copy()
    print(f"Split cutoff date: {pd.Timestamp(cutoff_date).date()}")
    print(f"Train: {len(train_df)} rows ({train_df['Date'].min().date()} -> {train_df['Date'].max().date()})")
    print(f"Test:  {len(test_df)} rows ({test_df['Date'].min().date()} -> {test_df['Date'].max().date()})")
    return train_df, test_df


def encode_categoricals(train_df, test_df, cat_cols):
    """Fit LabelEncoders on train only, apply to both. Any category seen
    only in test gets mapped to a fallback 'unknown' bucket instead of
    crashing."""
    encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        train_df[col] = le.fit_transform(train_df[col].astype(str))
        encoders[col] = le

        known = set(le.classes_)
        test_df[col] = test_df[col].astype(str).apply(lambda v: v if v in known else "__unseen__")
        if "__unseen__" in test_df[col].unique():
            # extend encoder with an explicit unseen bucket
            le.classes_ = np.append(le.classes_, "__unseen__")
        test_df[col] = le.transform(test_df[col])

        train_df[col] = train_df[col].astype("category")
        test_df[col] = test_df[col].astype("category")

    return train_df, test_df, encoders


def train_one_model(X_train, y_train, X_test, y_test, categorical_cols, objective="regression", alpha=None):
    """Train a single LightGBM model with early stopping. objective is
    either 'regression' (mean) or 'quantile' (needs alpha in (0, 1))."""
    params = dict(
        n_estimators=2000,
        learning_rate=0.05,
        random_state=RANDOM_STATE,
        objective=objective,
    )
    eval_metric = "mae"
    if objective == "quantile":
        params["alpha"] = alpha
        eval_metric = "quantile"

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric=eval_metric,
        categorical_feature=categorical_cols,
        callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=100)],
    )
    return model


def pinball_loss(y_true, y_pred, alpha):
    diff = y_true - y_pred
    return np.mean(np.maximum(alpha * diff, (alpha - 1) * diff))


def main():
    df = load_and_engineer(CSV_PATH)
    df = add_lag_rolling_features(df)
    train_df, test_df = time_based_split(df, TEST_FRACTION)

    # ---- Categorical encoding (fit on train only) ----
    train_df, test_df, encoders = encode_categoricals(train_df, test_df, CATEGORICAL_COLS)

    # ---- Optional scaler for downstream use (e.g. your RL env) ----
    # Fit on train only. NOT used for this model's own training.
    scaler = StandardScaler()
    scaler.fit(train_df[NUMERIC_COLS_FOR_SCALER])

    # ---- Features / target ----
    drop_cols = [TARGET_COL, "Date"]
    feature_cols = [c for c in train_df.columns if c not in drop_cols]

    X_train, y_train = train_df[feature_cols], train_df[TARGET_COL]
    X_test, y_test = test_df[feature_cols], test_df[TARGET_COL]

    # =========================================================
    # Mean-regression model (point forecast)
    # =========================================================
    print("\n" + "=" * 60)
    print("Training mean-regression model")
    print("=" * 60)
    mean_model = train_one_model(X_train, y_train, X_test, y_test, CATEGORICAL_COLS, objective="regression")

    y_pred_mean = mean_model.predict(X_test, num_iteration=mean_model.best_iteration_)
    mae = mean_absolute_error(y_test, y_pred_mean)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_mean))
    mape = np.mean(np.abs((y_test - y_pred_mean) / np.clip(y_test, 1, None))) * 100

    print(f"\nBest iteration: {mean_model.best_iteration_}")
    print(f"Test MAE:  {mae:.2f} units")
    print(f"Test RMSE: {rmse:.2f} units")
    print(f"Test MAPE: {mape:.2f}%")
    print("\nFirst 5 predictions vs actuals:")
    for p, a in zip(y_pred_mean[:5], y_test.values[:5]):
        print(f"  predicted: {p:.2f} | actual: {a:.2f}")

    # ---- Plots ----
    plt.figure(figsize=(10, 5))
    plt.plot(y_test.values[:50], label="Actual", marker="o")
    plt.plot(y_pred_mean[:50], label="Predicted (mean)", marker="x")
    plt.legend()
    plt.title("Actual vs Predicted Units Sold (first 50 test rows)")
    plt.tight_layout()
    plt.savefig("actual_vs_predicted.png", dpi=150)
    plt.close()

    plt.figure(figsize=(8, 6))
    lgb.plot_importance(mean_model, max_num_features=10)
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150)
    plt.close()

    joblib.dump(mean_model, OUTPUT_MEAN_MODEL_PATH)

    # =========================================================
    # Quantile-regression model(s) (safety-stock-oriented forecast)
    # =========================================================
    quantile_preds = {}
    for alpha in QUANTILE_ALPHAS:
        print("\n" + "=" * 60)
        print(f"Training quantile-regression model (alpha={alpha})")
        print("=" * 60)
        q_model = train_one_model(X_train, y_train, X_test, y_test, CATEGORICAL_COLS,
                                   objective="quantile", alpha=alpha)

        y_pred_q = q_model.predict(X_test, num_iteration=q_model.best_iteration_)
        quantile_preds[alpha] = y_pred_q

        coverage = np.mean(y_test.values <= y_pred_q) * 100
        loss = pinball_loss(y_test.values, y_pred_q, alpha)

        print(f"\nBest iteration: {q_model.best_iteration_}")
        print(f"Pinball loss (alpha={alpha}): {loss:.3f}")
        print(f"Coverage: {coverage:.1f}% of actual sales were <= the predicted "
              f"{int(alpha * 100)}th percentile (target: {alpha * 100:.0f}%)")
        print(f"Mean predicted q{alpha}: {y_pred_q.mean():.2f} vs mean predicted (mean model): {y_pred_mean.mean():.2f}")

        joblib.dump(q_model, OUTPUT_QUANTILE_MODEL_PATH_TMPL.format(alpha=alpha))

    # ---- Comparison plot: mean vs quantile forecast on first 50 test rows ----
    plt.figure(figsize=(10, 5))
    plt.plot(y_test.values[:50], label="Actual", marker="o")
    plt.plot(y_pred_mean[:50], label="Predicted (mean)", marker="x")
    for alpha, y_pred_q in quantile_preds.items():
        plt.plot(y_pred_q[:50], label=f"Predicted (q{alpha})", marker="^", linestyle="--")
    plt.legend()
    plt.title("Mean vs Quantile Forecast (first 50 test rows)")
    plt.tight_layout()
    plt.savefig("mean_vs_quantile_forecast.png", dpi=150)
    plt.close()

    # ---- Save shared artifacts ----
    joblib.dump(encoders, OUTPUT_ENCODERS_PATH)
    joblib.dump(scaler, OUTPUT_SCALER_PATH)
    joblib.dump(feature_cols, OUTPUT_FEATURE_ORDER_PATH)

    print("\nSaved models:", OUTPUT_MEAN_MODEL_PATH,
          *[OUTPUT_QUANTILE_MODEL_PATH_TMPL.format(alpha=a) for a in QUANTILE_ALPHAS])
    print("Saved:", OUTPUT_ENCODERS_PATH, OUTPUT_SCALER_PATH, OUTPUT_FEATURE_ORDER_PATH)
    print("Saved plots: actual_vs_predicted.png, feature_importance.png, mean_vs_quantile_forecast.png")


if __name__ == "__main__":
    main()

Dropped 1400 rows with insufficient history for lag/rolling features (expected: ~14 rows per store-product series, only at the start of each series).
Split cutoff date: 2023-08-11
Train: 57300 rows (2022-01-15 -> 2023-08-10)
Test:  14400 rows (2023-08-11 -> 2024-01-01)

Training mean-regression model
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008370 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2832
[LightGBM] [Info] Number of data points in the train set: 57300, number of used features: 23
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score 136.653316
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 7.26033	valid_0's l2: 71.9725
Early stopping, best iteration is:
[139]	valid_0's l1: 7.24171	valid_0's l2: 71.509

<Figure size 576x432 with 0 Axes>

In [8]:
"""
Reinforcement-learning replenishment policy (PPO), built on top of the
LightGBM demand model trained by train_lgbm_demand_model.py.

WHAT THIS SCRIPT DOES
----------------------
Trains a PPO agent (stable-baselines3) whose action is "how many units to
order today" for a store-product pair. The environment simulates the
consequence of that order using your already-trained LightGBM mean-demand
model as the demand generator, then scores the day with a standard
inventory cost reward. One POLICY is shared across all store-product
pairs (Store ID / Product ID are part of the observation), matching how
the demand model itself was trained on the pooled dataset.

KEY DESIGN CHOICES (per your answers)
--------------------------------------
  - Algorithm: PPO, continuous action space (order quantity, as a
    fraction of a per-step order cap -- see ReplenishmentEnv below).
  - Scope: one shared policy across all store-product pairs.
  - Reward: reward = -(holding_cost * ending_inventory)
                      -(stockout_cost * unmet_demand)
                      -(order_cost   * units_ordered)
    HOLDING_COST / STOCKOUT_COST / ORDER_COST below are illustrative
    defaults, not fitted to your business -- edit them in CONFIG.
  - This script only builds and (if run) trains the pipeline; it expects
    the artifacts produced by train_lgbm_demand_model.py to already
    exist on disk (paths in CONFIG below).

IMPORTANT CAVEAT: "Units Sold" in the source data is realized SALES, not
true demand -- it's implicitly capped at available inventory
(sales = min(demand, stock)). Wherever the historical data had a
stockout, the LightGBM model was trained on an already-truncated demand
value, so as a *simulator* it will tend to underestimate true demand in
stockout-heavy regimes. The RL agent is trained against this same model,
so it will be well-calibrated to the observed sales dynamics but may
stay mildly optimistic about peak demand. Standard mitigation: retrain
the demand model periodically on fresh data collected under the RL
policy's (better-stocked) behavior. Not implemented here.

ASSUMPTION ABOUT THE "Inventory Level" COLUMN: treated as stock
available for sale at the *start* of the day, before that day's sales
are subtracted -- i.e. exactly what we simulate as
`available_inventory = starting_inventory + order`. If your data
actually logs end-of-day inventory, swap the override in
`_predict_demand` accordingly.

Usage:
    pip install stable-baselines3 gymnasium
    python train_rl_replenishment.py
"""
import random
from collections import deque

import joblib
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor

# ------------------------------------------------------------------
# 0. Config
# ------------------------------------------------------------------
CSV_PATH = "C:\\Users\\a0952\\Desktop\\數創\\archive (1)\\retail_store_inventory.csv"

# Artifacts produced by train_lgbm_demand_model.py -- must already exist.
MEAN_MODEL_PATH = "lgb_sales_model_mean.pkl"
QUANTILE_MODEL_PATH = "lgb_sales_model_q0.9.pkl"   # used only as an eval baseline
ENCODERS_PATH = "label_encoders.pkl"
FEATURE_ORDER_PATH = "feature_order.pkl"

GROUP_COLS = ["Store ID", "Product ID"]
CATEGORICAL_COLS = ["Store ID", "Product ID", "Category", "Region",
                     "Weather Condition", "Seasonality"]
TARGET_COL = "Units Sold"

# Columns whose value the environment overrides every step because they
# depend on the agent's own actions / simulated history, rather than
# being replayed verbatim from the historical row.
STATE_DEPENDENT_COLS = ["Inventory Level", "Sales_Lag_1", "Sales_Lag_7",
                         "Sales_RollMean_7", "Sales_RollStd_7",
                         "Sales_RollMean_14", "Inventory_Lag_1"]

# ---- Cost model (EDIT THESE to your real unit economics) ----
HOLDING_COST_PER_UNIT = 0.5     # cost per unit left in inventory at day end
STOCKOUT_COST_PER_UNIT = 3.0    # cost per unit of unmet demand
ORDER_COST_PER_UNIT = 0.2       # cost per unit ordered (procurement/shipping)
ORDER_CAP_MULTIPLIER = 3.0      # order cap = this * recent rolling-mean demand
MIN_ORDER_CAP = 5.0             # floor so low-demand series still get room to order
DEMAND_NOISE_STD_FRAC = 0.15    # simulated demand noise, as a fraction of predicted mean

EPISODE_LENGTH = 30             # simulated days per training episode
N_ENVS = 4                      # parallel envs (DummyVecEnv)
TOTAL_TIMESTEPS = 200_000
SEED = 42

OUTPUT_MODEL_PATH = "ppo_replenishment_model"
OUTPUT_VECNORM_PATH = "vecnormalize.pkl"
OUTPUT_EVAL_PLOT_PATH = "rl_vs_baseline_cost.png"


# ------------------------------------------------------------------
# 1. Reproduce the exact feature engineering from train_lgbm_demand_model.py
#    (kept in sync manually -- if you change lag/rolling logic there,
#    mirror the change here so the simulator's features line up with
#    what the demand model was trained on).
# ------------------------------------------------------------------
def load_and_engineer(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(GROUP_COLS + ["Date"]).reset_index(drop=True)
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["DayOfWeek"] = df["Date"].dt.dayofweek
    return df


def add_lag_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(GROUP_COLS, observed=True)["Units Sold"]
    df["Sales_Lag_1"] = g.shift(1)
    df["Sales_Lag_7"] = g.shift(7)
    df["Sales_RollMean_7"] = g.transform(lambda s: s.shift(1).rolling(7).mean())
    df["Sales_RollStd_7"] = g.transform(lambda s: s.shift(1).rolling(7).std())
    df["Sales_RollMean_14"] = g.transform(lambda s: s.shift(1).rolling(14).mean())
    inv_g = df.groupby(GROUP_COLS, observed=True)["Inventory Level"]
    df["Inventory_Lag_1"] = inv_g.shift(1)
    df = df.dropna(subset=[
        "Sales_Lag_1", "Sales_Lag_7", "Sales_RollMean_7",
        "Sales_RollStd_7", "Sales_RollMean_14", "Inventory_Lag_1",
    ]).reset_index(drop=True)
    return df


def apply_saved_encoders(df: pd.DataFrame, encoders: dict, cat_cols: list) -> pd.DataFrame:
    """Apply the LabelEncoders fit during LightGBM training (never re-fit
    here -- that would silently change the model's category mapping)."""
    for col in cat_cols:
        le = encoders[col]
        known = set(le.classes_)
        vals = df[col].astype(str).apply(lambda v: v if v in known else "__unseen__")
        if "__unseen__" in vals.unique() and "__unseen__" not in known:
            le.classes_ = np.append(le.classes_, "__unseen__")
        df[col] = le.transform(vals)
        df[col] = df[col].astype("category")
    return df


def build_history_buffers(raw_df: pd.DataFrame) -> dict:
    """For each store-product group, a Date-indexed Series of raw Units
    Sold, used only to seed the 14-day sales buffer at episode reset."""
    buffers = {}
    for key, g in raw_df.groupby(GROUP_COLS, observed=True):
        buffers[key] = g.set_index("Date")["Units Sold"].sort_index()
    return buffers


# ------------------------------------------------------------------
# 2. Gymnasium environment
# ------------------------------------------------------------------
class ReplenishmentEnv(gym.Env):
    """One shared-policy environment: on each reset() a random
    store-product series and a random EPISODE_LENGTH-day window are
    sampled. Exogenous columns (price, weather, calendar, etc.) are
    replayed from the real historical rows; inventory/sales-history
    columns are simulated based on the agent's own order decisions,
    using the LightGBM demand model as the demand generator.
    """

    metadata = {"render_modes": []}

    def __init__(self, feat_df, feature_cols, sales_history, demand_model,
                 episode_starts, seed=None):
        super().__init__()
        self.feat_df = feat_df
        self.feature_cols = feature_cols
        self.sales_history = sales_history
        self.demand_model = demand_model
        self.episode_starts = episode_starts  # list of (group_key, start_row_idx)
        self._rng = np.random.default_rng(seed)

        self.action_space = spaces.Box(low=0.0, high=1.0, shape=(1,), dtype=np.float32)
        self.observation_space = spaces.Box(
            low=-1e6, high=1e6, shape=(len(feature_cols),), dtype=np.float32
        )

        # episode state, set in reset()
        self.group_key = None
        self.row_indices = None
        self.t = 0
        self.current_inventory = 0.0
        self.buffer = None

    # -- helpers -----------------------------------------------------
    def _row_template(self, row_idx):
        """A single-row DataFrame with correct dtypes (categoricals kept
        as pandas 'category'), copied from the historical row so all
        exogenous columns are already correct."""
        return self.feat_df.loc[[row_idx], self.feature_cols].copy()

    def _rolling_stats(self):
        buf = list(self.buffer)
        last7 = buf[-7:]
        return {
            "Sales_Lag_1": buf[-1],
            "Sales_Lag_7": buf[-7],
            "Sales_RollMean_7": float(np.mean(last7)),
            "Sales_RollStd_7": float(np.std(last7, ddof=1)) if len(last7) > 1 else 0.0,
            "Sales_RollMean_14": float(np.mean(buf)),
        }

    def _obs_from_row(self, row_idx):
        row_df = self._row_template(row_idx)
        stats = self._rolling_stats()
        row_df.loc[:, "Inventory Level"] = self.current_inventory
        row_df.loc[:, "Inventory_Lag_1"] = self.current_inventory
        for k, v in stats.items():
            row_df.loc[:, k] = v
        row = row_df.iloc[0].copy()
        for c in CATEGORICAL_COLS:
            row[c] = int(row[c])
        return row.values.astype(np.float32)

    def _order_cap(self):
        stats = self._rolling_stats()
        return max(MIN_ORDER_CAP, ORDER_CAP_MULTIPLIER * max(stats["Sales_RollMean_14"], 0.0))

    def _predict_demand(self, row_idx, available_inv):
        row_df = self._row_template(row_idx)
        stats = self._rolling_stats()
        row_df.loc[:, "Inventory Level"] = available_inv
        row_df.loc[:, "Inventory_Lag_1"] = self.current_inventory
        for k, v in stats.items():
            row_df.loc[:, k] = v
        best_iter = getattr(self.demand_model, "best_iteration_", None)
        pred = self.demand_model.predict(row_df, num_iteration=best_iter)
        return float(max(pred[0], 0.0))

    # -- gym API -------------------------------------------------------
    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        idx = self._rng.integers(0, len(self.episode_starts))
        self.group_key, start_row_idx = self.episode_starts[idx]
        self.row_indices = list(range(start_row_idx, start_row_idx + EPISODE_LENGTH))
        self.t = 0

        start_date = self.feat_df.loc[start_row_idx, "Date"]
        hist = self.sales_history[self.group_key]
        prior = hist[hist.index < start_date].tail(14)
        # Guaranteed >=14 values because episode_starts only includes
        # rows the lag/rolling step already validated as having 14 days
        # of prior history.
        self.buffer = deque(prior.values.astype(float), maxlen=14)
        self.current_inventory = float(self.feat_df.loc[start_row_idx, "Inventory_Lag_1"])

        obs = self._obs_from_row(self.row_indices[0])
        return obs, {}

    def step(self, action):
        order_units = float(np.clip(action[0], 0.0, 1.0)) * self._order_cap()
        row_idx = self.row_indices[self.t]

        available_inv = self.current_inventory + order_units
        demand_mean = self._predict_demand(row_idx, available_inv)
        noise = self._rng.normal(0.0, DEMAND_NOISE_STD_FRAC * max(demand_mean, 1.0))
        demand = max(0.0, demand_mean + noise)

        actual_sales = min(demand, available_inv)
        stockout = max(demand - available_inv, 0.0)
        ending_inv = available_inv - actual_sales

        reward = -(
            HOLDING_COST_PER_UNIT * ending_inv
            + STOCKOUT_COST_PER_UNIT * stockout
            + ORDER_COST_PER_UNIT * order_units
        )

        self.buffer.append(actual_sales)
        self.current_inventory = ending_inv
        self.t += 1
        terminated = self.t >= EPISODE_LENGTH

        info = {
            "order_units": order_units,
            "demand": demand,
            "actual_sales": actual_sales,
            "stockout": stockout,
            "ending_inventory": ending_inv,
            "holding_cost": HOLDING_COST_PER_UNIT * ending_inv,
            "stockout_cost": STOCKOUT_COST_PER_UNIT * stockout,
            "order_cost": ORDER_COST_PER_UNIT * order_units,
        }

        if terminated:
            obs = np.zeros(self.observation_space.shape, dtype=np.float32)
        else:
            obs = self._obs_from_row(self.row_indices[self.t])
        return obs, reward, terminated, False, info


def build_episode_starts(feat_df: pd.DataFrame) -> list:
    """All valid (group_key, start_row_idx) pairs with >=EPISODE_LENGTH
    contiguous rows remaining in that group. Must be called BEFORE the
    group columns are label-encoded, so group_key matches the raw
    string keys used by sales_history (see build_history_buffers)."""
    starts = []
    for key, g in feat_df.groupby(GROUP_COLS, observed=True):
        idxs = g.index.to_numpy()
        n_valid = len(idxs) - EPISODE_LENGTH
        if n_valid > 0:
            starts.extend((key, int(i)) for i in idxs[:n_valid])
    return starts


# ------------------------------------------------------------------
# 3. Baseline policy for evaluation: order up to the LightGBM q0.9
#    forecast (a natural "safety stock" heuristic already produced by
#    train_lgbm_demand_model.py). Compared against the RL policy so you
#    can see whether RL actually beats the heuristic you already have.
# ------------------------------------------------------------------
def run_baseline_episode(env: ReplenishmentEnv, quantile_model):
    obs, _ = env.reset()
    total_cost = {"holding": 0.0, "stockout": 0.0, "order": 0.0}
    for _ in range(EPISODE_LENGTH):
        row_idx = env.row_indices[env.t]
        row_df = env._row_template(row_idx)
        stats = env._rolling_stats()
        row_df.loc[:, "Inventory Level"] = env.current_inventory
        row_df.loc[:, "Inventory_Lag_1"] = env.current_inventory
        for k, v in stats.items():
            row_df.loc[:, k] = v
        best_iter = getattr(quantile_model, "best_iteration_", None)
        q90 = float(max(quantile_model.predict(row_df, num_iteration=best_iter)[0], 0.0))
        target_order = max(0.0, q90 - env.current_inventory)
        cap = env._order_cap()
        action = np.array([min(target_order / cap, 1.0) if cap > 0 else 0.0], dtype=np.float32)
        obs, reward, terminated, _, info = env.step(action)
        total_cost["holding"] += info["holding_cost"]
        total_cost["stockout"] += info["stockout_cost"]
        total_cost["order"] += info["order_cost"]
        if terminated:
            break
    return total_cost


def run_policy_episode(env: ReplenishmentEnv, model, vecnorm=None):
    obs, _ = env.reset()
    total_cost = {"holding": 0.0, "stockout": 0.0, "order": 0.0}
    for _ in range(EPISODE_LENGTH):
        obs_in = vecnorm.normalize_obs(obs) if vecnorm is not None else obs
        action, _ = model.predict(obs_in, deterministic=True)
        obs, reward, terminated, _, info = env.step(action)
        total_cost["holding"] += info["holding_cost"]
        total_cost["stockout"] += info["stockout_cost"]
        total_cost["order"] += info["order_cost"]
        if terminated:
            break
    return total_cost


# ------------------------------------------------------------------
# 4. Main
# ------------------------------------------------------------------
def main():
    random.seed(SEED)
    np.random.seed(SEED)

    print("Loading LightGBM artifacts...")
    mean_model = joblib.load(MEAN_MODEL_PATH)
    quantile_model = joblib.load(QUANTILE_MODEL_PATH)
    encoders = joblib.load(ENCODERS_PATH)
    feature_cols = joblib.load(FEATURE_ORDER_PATH)

    print("Rebuilding engineered features (must match train_lgbm_demand_model.py)...")
    raw_df = load_and_engineer(CSV_PATH)
    sales_history = build_history_buffers(raw_df)
    feat_df = add_lag_rolling_features(raw_df.copy())

    # IMPORTANT: compute episode start points (grouped by raw Store ID /
    # Product ID string values) BEFORE label-encoding those columns.
    # sales_history is also keyed by the raw string values (built from
    # raw_df above) -- encoding first would make the group keys integers
    # and silently break every sales_history[group_key] lookup at reset().
    episode_starts = build_episode_starts(feat_df)
    n_groups = feat_df.groupby(GROUP_COLS, observed=True).ngroups
    print(f"{len(episode_starts)} valid episode start points across {n_groups} store-product series.")

    feat_df = apply_saved_encoders(feat_df, encoders, CATEGORICAL_COLS)

    missing = [c for c in feature_cols if c not in feat_df.columns]
    if missing:
        raise ValueError(
            f"feature_order.pkl expects columns not present after re-engineering: {missing}. "
            "Check that this script's load_and_engineer/add_lag_rolling_features "
            "still match train_lgbm_demand_model.py."
        )

    # 90/10 split of episode starts for train / held-out eval, so the RL
    # agent isn't evaluated on windows it was directly trained on.
    rng = np.random.default_rng(SEED)
    shuffled = episode_starts.copy()
    rng.shuffle(shuffled)
    cutoff = int(0.9 * len(shuffled))
    train_starts, eval_starts = shuffled[:cutoff], shuffled[cutoff:]

    def make_env(seed_offset):
        def _init():
            env = ReplenishmentEnv(feat_df, feature_cols, sales_history,
                                    mean_model, train_starts, seed=SEED + seed_offset)
            return Monitor(env)
        return _init

    vec_env = DummyVecEnv([make_env(i) for i in range(N_ENVS)])
    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

    print("\n" + "=" * 60)
    print(f"Training PPO for {TOTAL_TIMESTEPS} timesteps")
    print("=" * 60)
    model = PPO("MlpPolicy", vec_env, verbose=1, seed=SEED, n_steps=EPISODE_LENGTH * 8,
                batch_size=64, gamma=0.99)
    model.learn(total_timesteps=TOTAL_TIMESTEPS)

    model.save(OUTPUT_MODEL_PATH)
    vec_env.save(OUTPUT_VECNORM_PATH)
    print(f"\nSaved model: {OUTPUT_MODEL_PATH}.zip")
    print(f"Saved VecNormalize stats: {OUTPUT_VECNORM_PATH}")

    # ---- Evaluation: RL policy vs. "order up to q0.9 forecast" baseline ----
    print("\n" + "=" * 60)
    print("Evaluating RL policy vs. quantile-forecast baseline on held-out episodes")
    print("=" * 60)
    eval_env = ReplenishmentEnv(feat_df, feature_cols, sales_history,
                                 mean_model, eval_starts, seed=SEED + 999)

    n_eval_episodes = min(50, len(eval_starts))
    rl_costs = {"holding": [], "stockout": [], "order": []}
    base_costs = {"holding": [], "stockout": [], "order": []}
    for _ in range(n_eval_episodes):
        rc = run_policy_episode(eval_env, model, vec_env)
        for k in rl_costs:
            rl_costs[k].append(rc[k])
        bc = run_baseline_episode(eval_env, quantile_model)
        for k in base_costs:
            base_costs[k].append(bc[k])

    def summarize(costs):
        totals = {k: np.mean(v) for k, v in costs.items()}
        totals["total"] = sum(totals.values())
        return totals

    rl_summary = summarize(rl_costs)
    base_summary = summarize(base_costs)
    print(f"\nRL policy       avg episode cost: {rl_summary}")
    print(f"Baseline (q0.9) avg episode cost: {base_summary}")

    labels = ["holding", "stockout", "order"]
    x = np.arange(len(labels))
    width = 0.35
    plt.figure(figsize=(8, 5))
    plt.bar(x - width / 2, [rl_summary[l] for l in labels], width, label="RL (PPO)")
    plt.bar(x + width / 2, [base_summary[l] for l in labels], width, label="Baseline (order-to-q0.9)")
    plt.xticks(x, labels)
    plt.ylabel(f"Avg cost per {EPISODE_LENGTH}-day episode")
    plt.title("RL policy vs. quantile-forecast baseline")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_EVAL_PLOT_PATH, dpi=150)
    plt.close()
    print(f"Saved comparison plot: {OUTPUT_EVAL_PLOT_PATH}")


if __name__ == "__main__":
    main()

Loading LightGBM artifacts...
Rebuilding engineered features (must match train_lgbm_demand_model.py)...
68700 valid episode start points across 100 store-product series.

Training PPO for 200000 timesteps
Using cpu device


KeyboardInterrupt: 

In [1]:
pip install -r requirements.txt

In [6]:
"""
train_rl.py

Trains one PPO agent (same network/hyperparameters) for each of the four
reward scenarios in reward_configs.SCENARIOS, on a train/eval split of the
100 (store, product) series.

Usage:
    python train_rl.py --scenario historical_baseline --timesteps 500000
    python train_rl.py --scenario all --timesteps 500000

Requires: gymnasium, stable-baselines3, torch, lightgbm, scikit-learn,
joblib, pandas, numpy (see requirements.txt). Not runnable as-is in a
sandbox without internet access -- see README.md for how this was validated.
"""

from __future__ import annotations

import argparse
import os

import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback

from feature_engineering import prepare_history
from demand_model import DemandModel
from replenishment_env import ReplenishmentEnv
from reward_configs import SCENARIOS

DATA_PATH = "retail_store_inventory.csv"
MODELS_DIR = "models"
EVAL_HOLDOUT_FRACTION = 0.2  # fraction of (store, product) pairs held out for eval
SEED = 42


def make_train_eval_pairs(history_df, seed=SEED):
    pairs = sorted(history_df.groupby(["Store ID", "Product ID"]).groups.keys())
    rng = np.random.default_rng(seed)
    rng.shuffle(pairs)
    n_eval = max(1, int(len(pairs) * EVAL_HOLDOUT_FRACTION))
    return pairs[n_eval:], pairs[:n_eval]  # train_pairs, eval_pairs


def make_env_fn(history_df, demand_model, scenario_name, pairs, seed):
    cfg = SCENARIOS[scenario_name]

    def _init():
        return ReplenishmentEnv(
            history_df, demand_model, cfg, rng_seed=seed, fixed_pairs=pairs
        )

    return _init


def train_scenario(scenario_name: str, timesteps: int):
    print(f"\n=== Training scenario: {scenario_name} ===")
    raw = pd.read_csv(DATA_PATH)
    history_df = prepare_history(raw)
    demand_model = DemandModel()

    train_pairs, eval_pairs = make_train_eval_pairs(history_df)

    train_env = DummyVecEnv([make_env_fn(history_df, demand_model, scenario_name, train_pairs, SEED)])
    train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

    eval_env = DummyVecEnv([make_env_fn(history_df, demand_model, scenario_name, eval_pairs, SEED + 1)])
    eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, training=False)

    out_dir = os.path.join(MODELS_DIR, scenario_name)
    os.makedirs(out_dir, exist_ok=True)

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=out_dir,
        log_path=out_dir,
        eval_freq=10_000,
        n_eval_episodes=len(eval_pairs),
        deterministic=True,
    )

    model = PPO(
        "MlpPolicy",
        train_env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=256,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        verbose=1,
        seed=SEED,
        tensorboard_log=os.path.join(out_dir, "tb"),
    )

    model.learn(total_timesteps=timesteps, callback=eval_callback)

    model.save(os.path.join(out_dir, "ppo_model"))
    train_env.save(os.path.join(out_dir, "vecnormalize.pkl"))
    print(f"Saved model + VecNormalize stats to {out_dir}/")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--scenario", choices=list(SCENARIOS.keys()) + ["all"], default="all"
    )
    parser.add_argument("--timesteps", type=int, default=500_000)
    args = parser.parse_args()

    scenarios = list(SCENARIOS.keys()) if args.scenario == "all" else [args.scenario]
    for s in scenarios:
        train_scenario(s, args.timesteps)


if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h]
                             [--scenario {historical_baseline,zero_stockout,high_holding_stockout_tolerant,pure_profit,all}]
                             [--timesteps TIMESTEPS]
ipykernel_launcher.py: error: unrecognized arguments: -f C:\Users\a0952\AppData\Roaming\jupyter\runtime\kernel-72b2dde4-8000-4a48-b43d-cc1a09f6145d.json


SystemExit: 2